# Variantes de BRAM-EV — un mécanisme interne à la fois

Ce notebook **n'exécute aucune simulation** : il lit les artefacts d'une
campagne déjà produite.

```bash
python main.py run --config experiments/ablation_variants.yaml
```

Là où `ablation.ipynb` demande *« qu'apporte l'ajout de ce composant ? »*, ce
notebook demande *« ce mécanisme doit-il fonctionner comme il fonctionne ? »*.
Chaque variante remplace **un seul** mécanisme interne de la méthode complète
et se compare à `bramev` :

| Variante | Mécanisme neutralisé | Remplacé par |
| --- | --- | --- |
| `bramev_nearest_offer` | utilité multicritère | le véhicule prend l'offre la plus proche |
| `bramev_fixed_alpha` | hétérogénéité des alpha | un alpha commun (`--alpha-fixed`) |
| `bramev_global_rep` | réputation par société | un score unique partagé |
| `bramev_event_score` | score proportionnel à la durée | une pénalité forfaitaire par événement |

Le sens de lecture est **BRAM-EV -> variante** : un écart défavorable signifie
que le mécanisme neutralisé était utile. Le plan est déclaré une seule fois,
dans `src/experiments/methods.py`.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path().resolve().parent))

import pandas as pd
from IPython.display import Image, display

import src.experiments.methods as methods
from src.pipeline import ablation, figures
from src.pipeline.params import CaseParams
from src.pipeline.store import RunStore

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)

## Choix du run

Il faut `bramev` **et** au moins une variante : sans la référence, aucun écart
n'est calculable. `latest_with_methods` prend le run le plus récent qui les
contient, et dit ce que contiennent les autres s'il n'en trouve aucun.

In [2]:
for path in RunStore.list_runs('../results_grid'):
    present = sorted({row['method'] for row in RunStore(path).read_summary()})
    print(f"{path.name}\n    {', '.join(present) or 'aucun cas'}")

20260823T180259Z_seed42_full-grid
    bramev, greedy
20260830T172116Z_seed42_ablation
    bramev, greedy, multistation, multistation_rep
20260901T060736Z_seed42_ablation-variants
    bramev, bramev_event_score, bramev_fixed_alpha, bramev_global_rep, bramev_nearest_offer


In [3]:
store = RunStore.latest_with_methods(('bramev',) + methods.VARIANTS,
                                     '../results_grid')
params = store.read_params()
manifest = store.read_manifest()

print(store.root)
print(params.describe())
print(f"graine={params.seed} | commit={manifest['git_commit']} | "
      f"cas={manifest['nb_cases_done']}/{manifest['nb_cases_planned']}")
print(f"alpha imposé aux variantes à alpha fixe : {params.alpha_fixed}")

../results_grid/20260901T060736Z_seed42_ablation-variants
seed=42 | scénarios=['optimistic', 'balance', 'pessimistic'] | flottes=[50, 100, 150, 200, 250] | méthodes=['bramev', 'bramev_nearest_offer', 'bramev_fixed_alpha', 'bramev_global_rep', 'bramev_event_score'] | 1440 slots (5.0 j) | 40 stations / 4 sociétés | 75 runs
graine=42 | commit=c9b8afbebca34621e2bbcd9e0a76193cc5d1005b | cas=75/75
alpha imposé aux variantes à alpha fixe : 0.5


## Ce qui est réellement neutralisé

Contrôle préalable : chaque variante doit différer de `bramev` par **exactement
un** mécanisme. Ces colonnes viennent des drapeaux effectivement appliqués
pendant la simulation (`src/pipeline/tables.py`), pas de l'intention déclarée.

In [4]:
summary = pd.read_csv(store.summary_path)

ORDRE = ['bramev'] + list(methods.VARIANTS)
summary['method'] = pd.Categorical(summary['method'], ORDRE + [
    m for m in summary['method'].unique() if m not in ORDRE], ordered=True)

variantes = summary[summary['method'].isin(ORDRE)].copy()

plan = (variantes[['method', 'method_label', 'method_family', 'offer_choice',
                   'alpha_mode', 'reputation_scope', 'score_weighting',
                   'broadcast', 'reputation', 'adaptation']]
        .drop_duplicates()
        .sort_values('method')
        .set_index('method'))
plan

,method_label,method_family,offer_choice,alpha_mode,reputation_scope,score_weighting,broadcast,reputation,adaptation
method,,,,,,,,,
bramev,BRAM-EV Full,ablation,utility,sampled,society,duration,True,True,True
bramev_nearest_offer,BRAM-EV / offre la plus proche,variant,nearest,sampled,society,duration,True,True,True
bramev_fixed_alpha,BRAM-EV / alpha fixe,variant,utility,fixed,society,duration,True,True,True
bramev_global_rep,BRAM-EV / réputation globale,variant,utility,sampled,global,duration,True,True,True
bramev_event_score,BRAM-EV / score par événement,variant,utility,sampled,society,event,True,True,True


In [5]:
champs = ['broadcast', 'reputation', 'adaptation', 'offer_choice',
          'alpha_mode', 'reputation_scope', 'score_weighting']
reference = plan.loc['bramev']

for nom in methods.VARIANTS:
    if nom not in plan.index:
        print(f"ABSENT    {nom}")
        continue
    change = [c for c in champs if plan.loc[nom, c] != reference[c]]
    etat = 'OK' if len(change) == 1 else 'ANOMALIE'
    print(f"{etat:9} {nom:22} change={', '.join(change):<20} "
          f"mécanisme={ablation.VARIANT_MECHANISM.get(nom, '?')}")

OK        bramev_nearest_offer   change=offer_choice         mécanisme=Utilité multicritère
OK        bramev_fixed_alpha     change=alpha_mode           mécanisme=Hétérogénéité des alpha
OK        bramev_global_rep      change=reputation_scope     mécanisme=Réputation par société
OK        bramev_event_score     change=score_weighting      mécanisme=Score proportionnel à la durée


In [6]:
# Décomposition calculée à la volée depuis summary.csv. Le pipeline persiste
# exactement les mêmes tables (`ablation.csv`, `ablation_mean.csv`) et les
# réécrit à chaque cas ; les recalculer ici rend le notebook utilisable sur une
# campagne encore en cours, interrompue, ou antérieure à l'étude d'ablation.
detail = pd.DataFrame(ablation.detail_rows(summary.to_dict('records')))
moyennes = pd.DataFrame(ablation.mean_rows(detail.to_dict('records')))

print(f"{len(detail)} écarts calculés sur "
      f"{detail[['scenario', 'nb_cars']].drop_duplicates().shape[0]} mondes")

900 écarts calculés sur 15 mondes


## La méthode complète face à chaque variante

Une ligne par méthode, moyenne sur tous les mondes du run. `bramev` est la
référence : lire les autres lignes comme des écarts à celle-ci.

In [7]:
METRIQUES = ['exact_satisfaction', 'rate_abs', 'mean_service_rate',
             'slot_waste_rate', 'nb_reservations', 'mean_waiting_time_min',
             'mean_travel_distance_km', 'mean_offers_per_demand',
             'total_ms_mean']

niveaux = (variantes.groupby('method', observed=True)[METRIQUES]
                    .mean()
                    .rename(index=methods.label)
                    .round(4))
niveaux

,exact_satisfaction,rate_abs,mean_service_rate,slot_waste_rate,nb_reservations,mean_waiting_time_min,mean_travel_distance_km,mean_offers_per_demand,total_ms_mean
method,,,,,,,,,
BRAM-EV Full,0.7806,0.1686,0.1047,0.4398,1454.4667,0.8292,0.3887,3.3111,15991.4809
BRAM-EV / offre la plus proche,0.7625,0.1691,0.1046,0.4385,1468.8667,5.1052,0.3802,3.2991,15728.6077
BRAM-EV / alpha fixe,0.7796,0.1689,0.1044,0.4398,1448.3333,0.7484,0.3872,3.2756,16063.7632
BRAM-EV / réputation globale,0.5153,0.1700,0.0618,0.4323,820.8667,0.4196,0.2981,0.6512,27956.8325
BRAM-EV / score par événement,0.8087,0.1662,0.1105,0.4393,1558.5333,0.9996,0.3471,5.1835,15310.3147


In [8]:
# Écart relatif à bramev, en pourcentage. Le signe est brut : la direction
# propre à chaque métrique est appliquée plus bas par `improvement`.
reference_valeurs = variantes[variantes['method'] == 'bramev'][METRIQUES].mean()
ecarts = (variantes.groupby('method', observed=True)[METRIQUES].mean()
          .div(reference_valeurs) - 1.) * 100.
ecarts.drop(index='bramev').rename(index=methods.label).round(2)

,exact_satisfaction,rate_abs,mean_service_rate,slot_waste_rate,nb_reservations,mean_waiting_time_min,mean_travel_distance_km,mean_offers_per_demand,total_ms_mean
method,,,,,,,,,
BRAM-EV / offre la plus proche,-2.32,0.28,-0.15,-0.30,0.99,515.68,-2.18,-0.36,-1.64
BRAM-EV / alpha fixe,-0.13,0.15,-0.31,-0.00,-0.42,-9.74,-0.39,-1.07,0.45
BRAM-EV / réputation globale,-33.98,0.80,-40.98,-1.71,-43.56,-49.40,-23.30,-80.33,74.82
BRAM-EV / score par événement,3.60,-1.43,5.54,-0.12,7.15,20.55,-10.69,56.55,-4.26


## Effet du mécanisme neutralisé

`ablation_mean.csv`, filtré sur `kind == 'variant'`. `share_improved` est la
part des mondes où **neutraliser** le mécanisme améliore la métrique : une
valeur basse est donc un argument *pour* le mécanisme.

In [9]:
print(ablation.render_mean_table(moyennes.to_dict('records')))

Composant                       Satisfaction exacte   Taux de no-show   Taux de service  Slots réservés perdus  Réservations confirmées
------------------------------  -------------------  ----------------  ----------------  ---------------------  -----------------------

Variantes de BRAM-EV (effet du mécanisme neutralisé)
Hétérogénéité des alpha                 -0.1% (33%)       +0.2% (27%)       -0.3% (27%)            -0.0% (60%)              -0.4% (20%)
Réputation par société                  -35.1% (0%)       +0.9% (47%)       -43.4% (0%)            -1.1% (67%)              -43.3% (0%)
Score proportionnel à la durée         +4.1% (100%)       -1.0% (53%)       +6.6% (80%)            -0.0% (47%)              +6.5% (73%)
Utilité multicritère                     -2.2% (0%)       +0.3% (27%)       -0.1% (27%)            -0.4% (67%)              +0.7% (67%)

Lecture : écart relatif moyen sur tous les mondes du run (part des mondes où le composant améliore la métrique).


In [10]:
bloc = moyennes[moyennes['kind'] == 'variant']

effets = bloc.pivot_table(index=['component', 'to_method'],
                          columns='metric_label',
                          values=['mean_delta_pct', 'share_improved'])
effets.round(3)

mean_delta_pct                                                                   \
metric_label                                        Besoins satisfaits Demandes rejetées Distance moyenne Latence bout-en-bout No-shows   
component                      to_method                                                                                                  
Hétérogénéité des alpha        bramev_fixed_alpha               -0.186             3.044           -0.339                0.356   -0.146   
Réputation par société         bramev_global_rep               -34.969           906.726          -22.548               78.045  -42.698   
Score proportionnel à la durée bramev_event_score                4.108           -72.940          -10.375               -2.388    5.340   
Utilité multicritère           bramev_nearest_offer             -1.428             1.905           -2.206               -0.779    1.006   

                                                                                                                                                              \
metric_label                                        Réservations confirmées Satisfaction exacte Slots réservés perdus Taux d'occupation Taux de confirmation   
component                      to_method                                                                                                                       
Hétérogénéité des alpha        bramev_fixed_alpha                    -0.364              -0.144                -0.003            -0.338               -1.254   
Réputation par société         bramev_global_rep                    -43.309             -35.085                -1.072           -44.310              -85.638   
Score proportionnel à la durée bramev_event_score                     6.537               4.062                -0.035             6.455               54.646   
Utilité multicritère           bramev_nearest_offer                   0.700              -2.231                -0.372            -0.405                0.547   

                                                                                                                          share_improved                     \
metric_label                                        Taux de no-show Taux de présentation Taux de service Temps de calcul Attente moyenne Besoins satisfaits   
component                      to_method                                                                                                                      
Hétérogénéité des alpha        bramev_fixed_alpha             0.214                0.071          -0.342           0.452           0.533                0.4   
Réputation par société         bramev_global_rep              0.876                0.931         -43.436          38.732           0.800                0.0   
Score proportionnel à la durée bramev_event_score            -1.009               -0.001           6.596           0.252           0.267                1.0   
Utilité multicritère           bramev_nearest_offer           0.307               -0.106          -0.130          -0.242           0.000                0.0   

                                                                                                                                              \
metric_label                                        Demandes rejetées Distance moyenne Latence bout-en-bout No-shows Réservations confirmées   
component                      to_method                                                                                                       
Hétérogénéité des alpha        bramev_fixed_alpha               0.200            0.600                0.333    0.333                   0.200   
Réputation par société         bramev_global_rep                0.000            1.000                0.000    1.000                   0.000   
Score proportionnel à la durée bramev_event_score               1.000            1.000                0.667    0.067           

In [11]:
# Verdict par mécanisme : sur combien de (monde x métrique) le neutraliser
# dégrade-t-il le résultat ? Une part élevée plaide pour le mécanisme.
bloc_detail = detail[detail['kind'] == 'variant']

verdict = (bloc_detail.groupby(['component', 'to_method'])
           .agg(comparaisons=('improvement', 'size'),
                neutraliser_dégrade=('improvement', lambda s: (~s).mean()))
           .round(3)
           .sort_values('neutraliser_dégrade', ascending=False))
verdict

,,comparaisons,neutraliser_dégrade
component,to_method,,
Réputation par société,bramev_global_rep,225,0.698
Hétérogénéité des alpha,bramev_fixed_alpha,225,0.649
Utilité multicritère,bramev_nearest_offer,225,0.627
Score proportionnel à la durée,bramev_event_score,225,0.324


## Dispersion par variante

Comme pour l'échelle, une moyenne peut être portée par un seul monde.

In [12]:
for metrique in ['exact_satisfaction', 'rate_abs', 'mean_service_rate']:
    sous = bloc_detail[bloc_detail['metric'] == metrique]
    if sous.empty:
        continue
    print(f"\n=== {sous['metric_label'].iloc[0]} — écart à bramev ===")
    stats = (sous.groupby('component')['delta']
                 .agg(['count', 'min', 'median', 'mean', 'max'])
                 .round(6))
    stats['mondes_améliorés'] = sous.groupby('component')['improvement'].mean().round(3)
    display(stats)


=== Satisfaction exacte — écart à bramev ===


,count,min,median,mean,max,mondes_améliorés
component,,,,,,
Hétérogénéité des alpha,15,-0.0056,-0.0002,-0.001000,0.0024,0.333
Réputation par société,15,-0.3258,-0.2773,-0.265240,-0.1715,0.000
Score proportionnel à la durée,15,0.0006,0.0035,0.028067,0.0913,1.000
Utilité multicritère,15,-0.0707,-0.0064,-0.018080,0.0000,0.000



=== Taux de no-show — écart à bramev ===


,count,min,median,mean,max,mondes_améliorés
component,,,,,,
Hétérogénéité des alpha,15,-0.0017,0.0003,0.000253,0.0029,0.267
Réputation par société,15,-0.0095,0.0007,0.001347,0.0187,0.467
Score proportionnel à la durée,15,-0.0106,-0.0001,-0.002420,0.0016,0.533
Utilité multicritère,15,-0.0020,0.0003,0.000473,0.0030,0.267



=== Taux de service — écart à bramev ===


,count,min,median,mean,max,mondes_améliorés
component,,,,,,
Hétérogénéité des alpha,15,-0.0024,-0.0002,-0.000327,0.0005,0.267
Réputation par société,15,-0.0798,-0.0384,-0.042920,-0.0145,0.000
Score proportionnel à la durée,15,-0.0006,0.0010,0.005807,0.0285,0.800
Utilité multicritère,15,-0.0028,-0.0001,-0.000160,0.0021,0.267


## Vérifications propres à chaque variante

Une variante peut afficher un écart nul simplement parce que son mécanisme
n'était pas sollicité. Les quatre contrôles ci-dessous distinguent
« mécanisme inutile » de « mécanisme jamais testé ».

### `bramev_nearest_offer` — le classement avait-il de quoi trancher ?

Choisir par utilité plutôt que par distance ne change rien si chaque demande ne
reçoit qu'une offre. `mean_offers_per_demand` dit si la comparaison a une
substance.

In [13]:
offres = (variantes.groupby('method', observed=True)['mean_offers_per_demand']
                   .mean().round(3))
print(offres, end='\n\n')

if offres.get('bramev', 0.) <= 1.:
    print("ATTENTION : au plus une offre par demande en moyenne — le critère "
          "de sélection n'a presque jamais eu à choisir. Un écart nul ne dit "
          "rien de l'utilité multicritère.")
else:
    print(f"{offres['bramev']:.2f} offre(s) par demande : le classement a "
          "effectivement eu à trancher.")

method
bramev                  3.311
bramev_nearest_offer    3.299
bramev_fixed_alpha      3.276
bramev_global_rep       0.651
bramev_event_score      5.184
Name: mean_offers_per_demand, dtype: float64

3.31 offre(s) par demande : le classement a effectivement eu à trancher.


### `bramev_fixed_alpha` — l'hétérogénéité a-t-elle été supprimée ?

La table `alpha` porte la trajectoire de l'arbitrage profit/risque par station.
Sous `bramev` les alpha sont dispersés et convergent par apprentissage
collectif ; sous `bramev_fixed_alpha` ils doivent tous valoir `alpha_fixed`,
ce qui rend l'apprentissage inerte — c'est voulu, et c'est ce qui isole la
contribution de l'hétérogénéité elle-même.

In [14]:
scenario, nb_cars = params.scenarios[-1], params.fleet_sizes[-1]

def lire_alpha(methode):
    chemin = store.table_path(CaseParams(scenario, nb_cars, methode), 'alpha')
    return pd.read_csv(chemin) if chemin.is_file() else None

for methode in ('bramev', 'bramev_fixed_alpha'):
    table = lire_alpha(methode)
    if table is None:
        print(f"{methode:20} (table absente)")
        continue
    initial = table[table['update_step'] == 0]['alpha']
    final = table[table['update_step'] == table['update_step'].max()]['alpha']
    print(f"{methode:20} étapes={table['update_step'].max() + 1:2}  "
          f"alpha initial: {initial.min():.3f}–{initial.max():.3f} "
          f"(écart-type {initial.std():.4f})  ->  "
          f"final: {final.min():.3f}–{final.max():.3f} "
          f"(écart-type {final.std():.4f})")

bramev               étapes=10  alpha initial: 0.104–0.899 (écart-type 0.2431)  ->  final: 0.222–0.697 (écart-type 0.1278)
bramev_fixed_alpha   étapes=10  alpha initial: 0.500–0.500 (écart-type 0.0000)  ->  final: 0.500–0.500 (écart-type 0.0000)


In [15]:
# Trajectoire moyenne et dispersion, étape d'apprentissage par étape.
trajectoires = {}
for methode in ('bramev', 'bramev_fixed_alpha'):
    table = lire_alpha(methode)
    if table is None:
        continue
    trajectoires[methode] = table.groupby('update_step')['alpha'].agg(
        moyenne='mean', ecart_type='std', minimum='min', maximum='max')

if trajectoires:
    display(pd.concat(trajectoires, axis=1).round(4))

bramev                            bramev_fixed_alpha                           
            moyenne ecart_type minimum maximum            moyenne ecart_type minimum maximum
update_step                                                                                 
0            0.5278     0.2431  0.1037  0.8990                0.5        0.0     0.5     0.5
1            0.5185     0.2233  0.1224  0.8507                0.5        0.0     0.5     0.5
2            0.5120     0.2005  0.1393  0.8178                0.5        0.0     0.5     0.5
3            0.4984     0.1855  0.1545  0.8005                0.5        0.0     0.5     0.5
4            0.4832     0.1726  0.1681  0.7783                0.5        0.0     0.5     0.5
5            0.4786     0.1604  0.1804  0.7644                0.5        0.0     0.5     0.5
6            0.4707     0.1492  0.1915  0.7428                0.5        0.0     0.5     0.5
7            0.4584     0.1416  0.2014  0.7233                0.5        0.0     0.5     0.5
8            0.4562     0.1344  0.2014  0.7131                0.5        0.0     0.5     0.5
9            0.4497     0.1278  0.2215  0.6970                0.5        0.0     0.5     0.5

### `bramev_global_rep` — le score est-il devenu un bien public ?

Sous `bramev`, un véhicule porte un score par société (`score_index` = la
société propriétaire) : une mauvaise réputation chez l'un ne se transmet pas à
l'autre. Sous `bramev_global_rep`, toutes les stations écrivent et lisent la
même case, donc `score_index` vaut 0 partout.

In [16]:
for methode in ('bramev', 'bramev_global_rep'):
    chemin = store.table_path(CaseParams(scenario, nb_cars, methode), 'stations')
    if not chemin.is_file():
        print(f"{methode:20} (table absente)")
        continue
    stations = pd.read_csv(chemin)
    print(f"{methode:20} score_index={sorted(stations['score_index'].unique())}  "
          f"pondération={sorted(stations['score_weighting'].unique())}  "
          f"sociétés={sorted(stations['society_id'].unique())}")

bramev               score_index=[np.int64(0), np.int64(1), np.int64(2), np.int64(3)]  pondération=['duration']  sociétés=[np.int64(0), np.int64(1), np.int64(2), np.int64(3)]
bramev_global_rep    score_index=[np.int64(0)]  pondération=['duration']  sociétés=[np.int64(0), np.int64(1), np.int64(2), np.int64(3)]


In [17]:
# Conséquence attendue : un score partagé se dégrade plus vite pour un même
# véhicule, donc les stations rejettent davantage et servent moins.
(variantes[variantes['method'].isin(['bramev', 'bramev_global_rep'])]
 .groupby('method', observed=True)
 .agg(demandes_rejetees=('nb_rejected_request', 'mean'),
      offres_par_demande=('mean_offers_per_demand', 'mean'),
      reservations=('nb_reservations', 'mean'),
      taux_de_service=('mean_service_rate', 'mean'))
 .round(3))

,demandes_rejetees,offres_par_demande,reservations,taux_de_service
method,,,,
bramev,7629.267,3.311,1454.467,0.105
bramev_global_rep,44598.800,0.651,820.867,0.062


### `bramev_event_score` — la durée pesait-elle vraiment ?

Une pénalité forfaitaire n'est différente d'une pénalité proportionnelle que si
les durées réservées varient. `d_prop` moyen et sa dispersion, lus dans la
table des acceptations, disent si l'écart entre les deux pondérations pouvait
se manifester.

In [18]:
chemin = store.table_path(CaseParams(scenario, nb_cars, 'bramev'), 'acceptances')
if chemin.is_file():
    acceptations = pd.read_csv(chemin)
    print(f"{len(acceptations)} offres acceptées")
    display(acceptations[['distance_km', 'waiting_time_min']].describe().round(3))
else:
    print('table acceptances absente (--no-save-tables ?)')

# Le rapport des pénalités vaut exactement d_n entre les deux pondérations
# (cf. Station.update_car_score) : plus les durées sont dispersées, plus les
# deux régimes divergent.
config = next(store.iter_results())['config']
print("\npoints de stratégie (société) :", config['base_points_strategy'])

2043 offres acceptées


,distance_km,waiting_time_min
count,2043.000,2043.000
mean,0.443,2.761
std,0.252,12.302
min,0.005,0.000
25%,0.262,0.000
50%,0.373,0.000
75%,0.592,0.000
max,0.997,195.000



points de stratégie (société) : {'pres': 4.0, 'abs': 3.0, 'late': 1.0, 'early': 0.5}


## Figures

In [19]:
chemin = store.figure_path('ablation_variants')
if chemin.is_file():
    print(chemin.name)
    display(Image(filename=str(chemin)))

In [20]:
figures.fig_ablation_variants(summary.to_dict('records'))

<Figure size 1510x440 with 4 Axes>

## Santé du run

In [21]:
print('invariants OK :', bool(summary['invariant_ok'].all()))
print('réservations non résolues :', int(summary['nb_unresolved'].sum()))
print('pannes :', int(summary['nb_breakdowns'].sum()))

vus = set()
for resultat in store.iter_results():
    for message in resultat['behaviors'].get('diagnostics', []):
        if message not in vus:
            vus.add(message)
            print(f"\n[diagnostic] {message}")

invariants OK : True
réservations non résolues : 84
pannes : 2432

[diagnostic] Aucune annulation anticipée réalisée alors que le scénario en prévoit : toutes sont reclassées en tardives. Une annulation ne peut être anticipée que si la réservation est prise suffisamment à l'avance ; avec un délai requête → arrivée quasi nul, la distinction anticipé / tardif n'est pas mesurable et la probabilité 'early' du scénario se réalise en 'late'. Voir 'reclassified' et 'median_lead_slots'.

[diagnostic] Délai moyen requête → arrivée prévue < 1 slot : les stations proposent des créneaux immédiats (rayon de recherche petit devant la vitesse par slot). Aucune réservation n'est prise à l'avance.
